In [ ]:
!pip3 install azure-devops dotenv

In [ ]:
# Import necessary libraries
from azure.devops.connection import Connection
from msrest.authentication import BasicAuthentication
import pprint
import re, json
from dotenv import load_dotenv
from azure.core.serialization import AzureJSONEncoder
import os

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Set up the Personal Access Token (PAT) and organization URL
personal_access_token = os.getenv('PERSONAL_ACCESS_TOKEN')
organization_url = os.getenv('ORGANIZATION_URL')
project_path = os.getenv('PROJECT_PATH')
print(f"Settings {organization_url} {personal_access_token[0:10]}")

In [ ]:
# Establish a connection to Azure DevOps
credentials = BasicAuthentication('', personal_access_token)
connection = Connection(base_url=organization_url, creds=credentials)

In [ ]:
def as_json(az_obj):
  if not az_obj:
    return {}
  as_string = json.dumps(az_obj.as_dict(), cls=AzureJSONEncoder)
  return json.loads(as_string)

from pydantic import BaseModel
from typing import List, Optional

class ThreadModel(BaseModel):
    status: str
    file_path: str
    line: int
    comments: str

def read_threads(connection, pr_url):
    match = re.search(r'https://dev.azure.com/(?P<org>.+?)/(?P<project>.+?)/_git/(?P<repo>.+?)/pullrequest/(?P<pr_id>.+?)$', pr_url)
    if not match:
        raise ValueError("Invalid pull request URL")

    organization, project, repo, pr_id = match.groups()
    git_client = connection.clients.get_git_client()
    threads = git_client.get_threads(
        repository_id=repo,
        pull_request_id=int(pr_id),
        project=project
    )
    return [ThreadModel(**{
        "status": thread.status,
        "file_path": thread.thread_context.file_path,
        "line": (thread.thread_context.right_file_start.line 
                 if thread.thread_context.right_file_start 
                 else thread.thread_context.left_file_start.line),
        "comments": " ".join(comment.content for comment in thread.comments)
    }) for thread in threads if thread.thread_context]

def insert_comments_into_files(threads, project_path):
    # Step 1: Group comments by file
    comments_by_file = {}
    for thread in threads:
        if thread.status.lower() == 'active':
            file_path = thread.file_path
            if file_path not in comments_by_file:
                comments_by_file[file_path] = []

            comments_by_file[file_path].append((thread.line, thread.comments))

    # Step 2: Insert comments into each file
    for file_path, comments_to_insert in comments_by_file.items():
        abs_file_path = os.path.join(project_path, file_path)
        with open(abs_file_path, 'r') as file:
            file_contents = file.readlines()

        comments_to_insert.sort()  # Ensure the comments are processed in line number order

        new_file_contents = []
        last_line = 0

        for line_num, content in comments_to_insert:
            # Check if line_num is greater than available lines
            if line_num - 1 > len(file_contents):
                last_line = len(file_contents)
                break
            new_file_contents.extend(file_contents[last_line:line_num - 1])  # Add original content up to the comment line
            new_file_contents.append(f'>{content}</codx>\n')  # Add the comment
            last_line = line_num - 1

        new_file_contents.extend(file_contents[last_line:])  # Add the remaining original content

        # Write back to the file
        with open(abs_file_path, 'w') as file:
            file.writelines(new_file_contents)
        print(f"Comments inserted in {file_path}")
    print("All active comments inserted successfully.")
# Example usage
get_pr_comments_from_url('https://dev.azure.com/world2meet/W2Fly/_git/app-mvn-mro-management-api/pullrequest/160928')